# 9 AM Monday. Four pipelines broken. 75 minutes on the wall clock. Triage, fix the top two, document the other two.

It is 9 AM Monday. You open Slack and find this in `#data-incidents`:

> **CFO (08:47):** revenue dashboard is showing yesterday's number as zero. need an update by 10am.
> **status-page (auto, 08:50):** All services operational.
> **finance-billing-alerts (08:52):** ⚠ provisioned-throughput exceeded on `customer-deltas` table, 47 retries in last hour.
> **eng-oncall:** Sam is on vacation through Tuesday. Slack DMs auto-replied.
> **CTO (08:55):** can someone update the customer notification SLA banner?

Four pipelines are down. The status page says everything is fine. The on-call engineer who knows two of them is on vacation. You have 75 minutes to triage by business impact, fix the two highest-impact pipelines, and write the handoff note for the other two. Your manager wants the post-mortem by tomorrow's standup.

This is the lab no cert teaches and no bootcamp simulates. The thing that separates a senior DE from a mid-level DE is calm under partial information. Today you build that muscle in a real AWS environment with four real broken pipelines.

The five deliverables map to five checkpoints:
1. The pre-broken environment is up; all four pipelines fail when triggered.
2. Priority-1 pipeline #1 (revenue aggregation) runs end-to-end after your fix.
3. Priority-1 pipeline #2 (Step Functions workflow) succeeds after your IAM fix.
4. `portfolio/handoff.md` exists with named root cause and recommended next step for the two pipelines you did NOT fix.
5. `portfolio/postmortem.md` exists with timeline, root cause, blast radius, follow-ups, and word count 150-400.

**Time.** About 90 minutes hands-on. The 75-minute display clock is the simulation; the kernel does not kill you at 75 minutes. The point of the wall clock is to practice triage under pressure, not to be killed by a timer.

**Cost.** This lab costs about $1.20 on a clean run. Glue ETL is the line item (one Glue script runs once successfully, plus a failed run during fix attempts). DynamoDB provisioned 1 RCU + 1 WCU is roughly 0.1 cents per hour. Kinesis 1 shard at $0.015/hour is the only steady drain. SNS, Step Functions, Lambda are free at lab volume. A realistic upper bound with reruns is $2.50.

**Critical resource.** The Kinesis Data Stream is critical-tagged. Cleanup tears it down first.

**Portfolio mode.** The cleanup cell preserves the `portfolio/` prefix in S3 so your handoff + post-mortem markdowns survive. Download links are surfaced via the companion panel.

This lab is part of the Labrun Data Engineering job-prep track, Data Reliability Engineering sub-track. It is a killer lab and a chaos-mode lab.


In [ ]:
!pip install --quiet labrun-checks==0.7.0 boto3


In [ ]:
# Imports and per-lab constants.

import atexit
import io
import json
import re
import sys
import time
import uuid
import zipfile
from getpass import getpass

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    CheckpointResult,
    CleanupResource,
    check,
    cleanup,
    register,
    run_cleanup,
)

LAB_SLUG = "oncall-pipeline-triage"
LAB_ID = LAB_SLUG
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_SLUG

REGION = "us-east-1"

BUCKET_NAME = None  # set in setup
DATABASE_NAME = f"labrun_{LAB_SLUG.replace('-', '_')}_db"

# Pipeline resource names
REVENUE_JOB = f"labrun-{LAB_SLUG}-revenue"
SFN_NAME = f"labrun-{LAB_SLUG}-sfn"
DDB_TABLE = f"labrun-{LAB_SLUG}-ddb-table"
CONSUMER_FN = f"labrun-{LAB_SLUG}-consumer"
STREAM_NAME = f"labrun-{LAB_SLUG}-stream"
SNS_TOPIC = f"labrun-{LAB_SLUG}-alerts"

GLUE_ROLE = f"labrun-{LAB_SLUG}-glue-role"
SFN_ROLE = f"labrun-{LAB_SLUG}-sfn-role"
CONSUMER_ROLE = f"labrun-{LAB_SLUG}-consumer-role"

REVENUE_TABLE = "revenue_daily"

# Runtime captured during setup
ACCOUNT_ID = None
STATE_MACHINE_ARN = None
SFN_ROLE_ARN = None
GLUE_ROLE_ARN = None
CONSUMER_ROLE_ARN = None
SNS_TOPIC_ARN = None
STREAM_ARN = None
ESM_UUID = None


In [ ]:
# NBVAL_SKIP
# Setup.

print("Paste your lab session token (copy from the Labrun lab page):")
SESSION_TOKEN = getpass("Session token: ").strip()
if not SESSION_TOKEN:
    raise SystemExit("Missing session token.")

print()
AWS_ACCESS_KEY_ID = getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
AWS_SESSION_TOKEN = getpass("AWS_SESSION_TOKEN (press Enter if your account does not use one): ").strip() or None

AWS_CREDENTIALS = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "aws_session_token": AWS_SESSION_TOKEN,
}

sts = boto3.client("sts", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    identity = sts.get_caller_identity()
except ClientError as exc:
    raise SystemExit(f"AWS credentials rejected: {exc}")

ACCOUNT_ID = identity["Account"]
BUCKET_NAME = f"labrun-{LAB_SLUG}-{ACCOUNT_ID}"
print(f"Account: {ACCOUNT_ID}; bucket: {BUCKET_NAME}")

register(SESSION_TOKEN, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print("Setup complete.")


In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST. Kinesis stream tears down first because it is the
# only hourly-billed resource. Cleanup PRESERVES portfolio/ prefix in S3
# so the handoff and post-mortem markdowns survive.

CLEANUP_MANIFEST = [
    CleanupResource(type="kinesis_stream", id=STREAM_NAME, region=REGION, critical=True,
        cli_delete_command=f"aws kinesis delete-stream --stream-name {STREAM_NAME} --enforce-consumer-deletion --region {REGION}"),
    CleanupResource(type="lambda_event_source_mapping", id="pending", region=REGION),
    CleanupResource(type="lambda_function", id=CONSUMER_FN, region=REGION,
        cli_delete_command=f"aws lambda delete-function --function-name {CONSUMER_FN} --region {REGION}"),
    CleanupResource(type="sfn_state_machine", id="pending", region=REGION),
    CleanupResource(type="glue_job", id=REVENUE_JOB, region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {REVENUE_JOB} --region {REGION}"),
    CleanupResource(type="sns_topic", id="pending", region=REGION),
    CleanupResource(type="dynamodb_table", id=DDB_TABLE, region=REGION,
        cli_delete_command=f"aws dynamodb delete-table --table-name {DDB_TABLE} --region {REGION}"),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{REVENUE_TABLE}", region=REGION),
    CleanupResource(type="iam_role", id=SFN_ROLE, region=REGION),
    CleanupResource(type="iam_role", id=GLUE_ROLE, region=REGION),
    CleanupResource(type="iam_role", id=CONSUMER_ROLE, region=REGION),
    CleanupResource(type="glue_database", id=DATABASE_NAME, region=REGION),
    CleanupResource(type="s3_bucket", id=BUCKET_NAME, region=REGION),
]


def _atexit_cleanup():
    print("[atexit] If cleanup did not run, your DMS-free portfolio costs are still ~$0 but Kinesis is alive at $0.015/hr; run the cleanup cell now.")


atexit.register(_atexit_cleanup)

# Orphan scan
tagging = boto3.client("resourcegroupstaggingapi", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    orphan_resp = tagging.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}])
    orphan_arns = [r["ResourceARN"] for r in orphan_resp.get("ResourceTagMappingList", [])]
except ClientError as exc:
    raise SystemExit(f"Orphan scan failed ({exc}). Check the labrun-tag-get-resources inline policy.")

if orphan_arns:
    print("Found orphans from a previous session. Cleanup is blocking.")
    for arn in orphan_arns:
        print(f"  ORPHAN: {arn}")
    raise SystemExit("Run the previous session's cleanup cell first.")

print("No orphans. Manifest declared. Proceed to Cell 5 to stand up the broken environment.")


## Standing up the broken environment

The next cell stands up four pre-broken pipelines plus the incident channel markdown. This is the helper cell; you do not edit it. After this cell completes you have:

- **Pipeline 1**: A Glue ETL job (`labrun-oncall-pipeline-triage-revenue`) that aggregates revenue per customer cohort. Broken by a divide-by-zero when one cohort has zero customers (the script does `revenue / customer_count` without a guard).
- **Pipeline 2**: A Step Functions state machine (`labrun-oncall-pipeline-triage-sfn`) that publishes an alert when revenue drops. Broken because the SFN execution role is missing `sns:Publish` on the topic.
- **Pipeline 3**: A DynamoDB table (`labrun-oncall-pipeline-triage-ddb-table`) at provisioned 1 RCU + 1 WCU. The CFO's customer-deltas batch tries to write 50 items at once; throttles.
- **Pipeline 4**: A Lambda consumer (`labrun-oncall-pipeline-triage-consumer`) wired to a 1-shard Kinesis stream. Broken because the Lambda's trust policy trusts `glue.amazonaws.com` instead of `lambda.amazonaws.com`; Lambda cannot invoke the function.

The incident channel markdown lives at `s3://<bucket>/incident-channel/2026-05-15-monday.md`.

Each pipeline is small on purpose. The chaos is the cognitive load of four simultaneous failures, not the volume of resources.


In [ ]:
# NBVAL_SKIP
# Setup helper: stand up the 4 broken pipelines + incident channel.

s3 = boto3.client("s3", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
iam = boto3.client("iam", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
glue = boto3.client("glue", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
ddb = boto3.client("dynamodb", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
sns = boto3.client("sns", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
sfn = boto3.client("stepfunctions", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
kinesis = boto3.client("kinesis", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
lambda_client = boto3.client("lambda", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
athena = boto3.client("athena", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})


def stand_up_broken_environment():
    global SFN_ROLE_ARN, GLUE_ROLE_ARN, CONSUMER_ROLE_ARN, SNS_TOPIC_ARN, STREAM_ARN
    global STATE_MACHINE_ARN, ESM_UUID

    # S3 bucket
    try:
        s3.create_bucket(Bucket=BUCKET_NAME)
        s3.put_bucket_tagging(Bucket=BUCKET_NAME, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})
    except ClientError as exc:
        if exc.response["Error"]["Code"] not in ("BucketAlreadyOwnedByYou",):
            raise
    print(f"  bucket ready: {BUCKET_NAME}")

    # Incident channel markdown
    INCIDENT_MD = '''# data-incidents -- 2026-05-15 (Monday)

08:47 CFO: revenue dashboard is showing yesterday's number as zero. need an update by 10am.

08:50 status-page (auto): All services operational.

08:52 finance-billing-alerts: PROVISIONED THROUGHPUT EXCEEDED on customer-deltas table. 47 retries in the last hour. Daily customer-snapshot job started at 08:30; first retries at 08:35.

08:53 monitoring-bot: SFN execution arn:aws:states:us-east-1:...:execution/labrun-oncall-pipeline-triage-sfn/<exec> FAILED. State "PublishRevenueAlert" raised States.TaskFailed.

08:55 CTO: can someone update the customer notification SLA banner? marketing site copy.

08:57 eng-oncall: Sam is on vacation through Tuesday. Slack DMs auto-replied. Their pipelines are the revenue Glue job and the Kinesis customer-events consumer.

09:01 platform-ops: noticed labrun-oncall-pipeline-triage-consumer lambda has been throwing AssumeRole errors since 02:00. cloudwatch logs show "User: arn:aws:sts::*:assumed-role/<role-name>/<session-name> is not authorized to perform: lambda:InvokeFunction". the consumer was last edited Friday by Sam.

09:03 finance-batch-job: customer-snapshot retried 50 more times. backlog of ~2500 customer deltas not landed yet.

09:10 you: triage time.
'''
    s3.put_object(Bucket=BUCKET_NAME, Key="incident-channel/2026-05-15-monday.md", Body=INCIDENT_MD.encode("utf-8"))
    print("  incident channel markdown written")

    # Glue database + Iceberg revenue table
    try:
        glue.create_database(DatabaseInput={"Name": DATABASE_NAME})
    except glue.exceptions.AlreadyExistsException:
        pass

    # Workgroup for Athena queries (so we have a result location)
    WG = f"labrun-{LAB_SLUG}-wg"
    try:
        athena.create_work_group(
            Name=WG,
            Configuration={"ResultConfiguration": {"OutputLocation": f"s3://{BUCKET_NAME}/athena/"}, "EnforceWorkGroupConfiguration": True},
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
    except Exception:
        pass

    # Bronze + Iceberg revenue table
    seed_rows = '\n'.join(json.dumps({
        "event_date": "2026-05-14",
        "cohort": cohort,
        "revenue": rev,
        "customer_count": cnt,
    }) for cohort, rev, cnt in [
        ("enterprise", 1000000, 50),
        ("smb", 250000, 500),
        ("self-serve-new", 5000, 0),  # divide-by-zero trap
        ("self-serve-existing", 75000, 1500),
    ])
    s3.put_object(Bucket=BUCKET_NAME, Key="bronze/cohorts.jsonl", Body=seed_rows.encode("utf-8"))

    def athena_q(query, timeout=60):
        qid = athena.start_query_execution(
            QueryString=query, WorkGroup=WG,
            QueryExecutionContext={"Database": DATABASE_NAME},
        )["QueryExecutionId"]
        deadline = time.time() + timeout
        while time.time() < deadline:
            s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
            if s == "SUCCEEDED":
                return qid
            if s in ("FAILED", "CANCELLED"):
                raise RuntimeError(f"Athena {qid} {s}")
            time.sleep(2)
        raise TimeoutError(f"Athena {qid} timed out")

    athena_q(
        f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{REVENUE_TABLE} ("
        f"  event_date string, cohort string, revenue_per_customer double, total_revenue bigint"
        f") LOCATION 's3://{BUCKET_NAME}/{REVENUE_TABLE}/' "
        f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
    )

    # Glue IAM role
    try:
        iam.create_role(
            RoleName=GLUE_ROLE,
            AssumeRolePolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "glue.amazonaws.com"}, "Action": "sts:AssumeRole"}]}),
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        iam.attach_role_policy(RoleName=GLUE_ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole")
        iam.put_role_policy(RoleName=GLUE_ROLE, PolicyName="labrun-glue-inline", PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Action": ["s3:*", "athena:*", "glue:*"], "Resource": "*"}],
        }))
    except iam.exceptions.EntityAlreadyExistsException:
        pass
    GLUE_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{GLUE_ROLE}"

    # BROKEN revenue script (divide-by-zero on the self-serve-new cohort)
    BROKEN_REVENUE_SCRIPT = '''
import sys, json
from awsglue.context import GlueContext
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType, LongType

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET", "DATABASE_NAME", "REVENUE_TABLE"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session

df = spark.read.json(f"s3://" + args["BUCKET"] + "/bronze/cohorts.jsonl")

# Bug: customer_count can be 0 for the self-serve-new cohort. The divide
# raises Py4JJavaError with java.lang.ArithmeticException at runtime when
# Spark materializes the column.
df_out = df.withColumn("revenue_per_customer", col("revenue") / col("customer_count")) \
           .withColumnRenamed("revenue", "total_revenue") \
           .select("event_date", "cohort", "revenue_per_customer", "total_revenue")

df_out.createOrReplaceTempView("delta")

spark.sql("MERGE INTO glue_catalog." + args["DATABASE_NAME"] + "." + args["REVENUE_TABLE"] + " t "
          "USING delta s ON t.event_date = s.event_date AND t.cohort = s.cohort "
          "WHEN MATCHED THEN UPDATE SET revenue_per_customer = s.revenue_per_customer, total_revenue = s.total_revenue "
          "WHEN NOT MATCHED THEN INSERT * ")
print("LABRUN_REVENUE_OK")
'''
    s3.put_object(Bucket=BUCKET_NAME, Key="scripts/revenue.py", Body=BROKEN_REVENUE_SCRIPT.encode("utf-8"))

    iceberg_conf = (
        "spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog "
        f"--conf spark.sql.catalog.glue_catalog.warehouse=s3://{BUCKET_NAME}/warehouse/ "
        "--conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog "
        "--conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO "
        "--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    print("  IAM propagation, hold for 15 seconds...")
    time.sleep(15)

    try:
        glue.create_job(
            Name=REVENUE_JOB,
            Role=GLUE_ROLE_ARN,
            Command={"Name": "glueetl", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/revenue.py", "PythonVersion": "3"},
            DefaultArguments={
                "--BUCKET": BUCKET_NAME,
                "--DATABASE_NAME": DATABASE_NAME,
                "--REVENUE_TABLE": REVENUE_TABLE,
                "--datalake-formats": "iceberg",
                "--conf": iceberg_conf,
                "--TempDir": f"s3://{BUCKET_NAME}/temp/",
            },
            GlueVersion="4.0",
            WorkerType="G.1X",
            NumberOfWorkers=2,
            Timeout=10,
            Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        )
        print("  pipeline 1: Glue ETL job created (divide-by-zero bug)")
    except glue.exceptions.AlreadyExistsException:
        pass

    # Pipeline 2: SNS topic + SFN role MISSING sns:Publish
    topic = sns.create_topic(Name=SNS_TOPIC, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
    SNS_TOPIC_ARN = topic["TopicArn"]

    try:
        iam.create_role(
            RoleName=SFN_ROLE,
            AssumeRolePolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "states.amazonaws.com"}, "Action": "sts:AssumeRole"}]}),
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        # BUG: only logs:* attached, no sns:Publish
        iam.put_role_policy(RoleName=SFN_ROLE, PolicyName="labrun-sfn-inline", PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Action": ["logs:*"], "Resource": "*"}],
        }))
    except iam.exceptions.EntityAlreadyExistsException:
        pass
    SFN_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{SFN_ROLE}"

    print("  IAM propagation #2, hold for 15 seconds...")
    time.sleep(15)

    asl = {
        "Comment": "Publish a revenue alert",
        "StartAt": "PublishRevenueAlert",
        "States": {
            "PublishRevenueAlert": {
                "Type": "Task",
                "Resource": "arn:aws:states:::sns:publish",
                "Parameters": {"TopicArn": SNS_TOPIC_ARN, "Message": "Revenue dropped overnight"},
                "End": True,
            }
        },
    }
    sm = sfn.create_state_machine(
        name=SFN_NAME,
        definition=json.dumps(asl),
        roleArn=SFN_ROLE_ARN,
        tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}],
    )
    STATE_MACHINE_ARN = sm["stateMachineArn"]
    print(f"  pipeline 2: SFN state machine created (missing sns:Publish)")

    # Pipeline 3: DynamoDB at provisioned 1/1
    try:
        ddb.create_table(
            TableName=DDB_TABLE,
            AttributeDefinitions=[{"AttributeName": "customer_id", "AttributeType": "S"}],
            KeySchema=[{"AttributeName": "customer_id", "KeyType": "HASH"}],
            ProvisionedThroughput={"ReadCapacityUnits": 1, "WriteCapacityUnits": 1},
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        waiter = ddb.get_waiter("table_exists")
        waiter.wait(TableName=DDB_TABLE)
        print("  pipeline 3: DynamoDB table created at 1 RCU / 1 WCU (throttle trap)")
    except ddb.exceptions.ResourceInUseException:
        pass

    # Pipeline 4: Kinesis stream + Lambda with BROKEN trust policy
    try:
        kinesis.create_stream(StreamName=STREAM_NAME, ShardCount=1, StreamModeDetails={"StreamMode": "PROVISIONED"}, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})
    except kinesis.exceptions.ResourceInUseException:
        pass
    print("  pipeline 4: Kinesis stream creating, hold ~60s...")
    deadline = time.time() + 180
    while time.time() < deadline:
        try:
            if kinesis.describe_stream_summary(StreamName=STREAM_NAME)["StreamDescriptionSummary"]["StreamStatus"] == "ACTIVE":
                break
        except Exception:
            pass
        time.sleep(5)
    STREAM_ARN = f"arn:aws:kinesis:{REGION}:{ACCOUNT_ID}:stream/{STREAM_NAME}"

    # BROKEN trust policy: principal is glue.amazonaws.com instead of lambda.amazonaws.com
    try:
        iam.create_role(
            RoleName=CONSUMER_ROLE,
            AssumeRolePolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "glue.amazonaws.com"}, "Action": "sts:AssumeRole"}]}),
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        iam.put_role_policy(RoleName=CONSUMER_ROLE, PolicyName="labrun-consumer-inline", PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{"Effect": "Allow", "Action": ["kinesis:*", "logs:*"], "Resource": "*"}],
        }))
    except iam.exceptions.EntityAlreadyExistsException:
        pass
    CONSUMER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{CONSUMER_ROLE}"

    print("  IAM propagation #3, hold for 15 seconds...")
    time.sleep(15)

    consumer_code = '''
def handler(event, context):
    return {"processed": len(event.get("Records", []))}
'''
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w") as z:
        z.writestr("index.py", consumer_code)
    buf.seek(0)
    zip_bytes = buf.read()
    # The Lambda CREATE will fail because Lambda cannot assume the role.
    # We catch and report; the BROKEN state is exactly this.
    try:
        lambda_client.create_function(
            FunctionName=CONSUMER_FN,
            Runtime="python3.13",
            Role=CONSUMER_ROLE_ARN,
            Handler="index.handler",
            Code={"ZipFile": zip_bytes},
            Timeout=30,
            Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        )
        print("  pipeline 4: Lambda created against broken trust policy (likely InvalidParameterValueException on first attempt)")
    except lambda_client.exceptions.InvalidParameterValueException as exc:
        print(f"  pipeline 4: Lambda create REJECTED as expected. {exc}")
        # We retry with the broken role; Lambda will eventually create
        # because IAM lets you create a function with a role you do not
        # have AssumeRole on. The function then fails to invoke.
        for _ in range(6):
            time.sleep(5)
            try:
                lambda_client.create_function(
                    FunctionName=CONSUMER_FN, Runtime="python3.13", Role=CONSUMER_ROLE_ARN,
                    Handler="index.handler", Code={"ZipFile": zip_bytes}, Timeout=30,
                    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
                )
                break
            except Exception:
                continue

    # Update manifest with the discovered runtime IDs.
    for r in CLEANUP_MANIFEST:
        if r.type == "sfn_state_machine":
            r.id = STATE_MACHINE_ARN
            r.cli_delete_command = f"aws stepfunctions delete-state-machine --state-machine-arn {STATE_MACHINE_ARN} --region {REGION}"
        if r.type == "sns_topic":
            r.id = SNS_TOPIC_ARN
            r.cli_delete_command = f"aws sns delete-topic --topic-arn {SNS_TOPIC_ARN} --region {REGION}"

    print()
    print("Broken environment up. Read the incident channel markdown and start your triage.")


stand_up_broken_environment()


## Task 1: Trigger all four pipelines and confirm each fails with the expected error class

Before you fix anything, prove the broken state. Read the incident channel markdown out of S3, then trigger each pipeline once and capture its failure. The checkpoint validates that you saw each broken pipeline fail in the expected way; it does not score WHICH triggers you ran, only that the failures match.

Helper triggers are provided. They wrap each pipeline's native invocation and return a status dict.


In [ ]:
# NBVAL_SKIP
# Task 1 helpers: trigger each pipeline, return its failure state.

logs_client = boto3.client("logs", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})


def trigger_pipeline_1():
    """Glue revenue job. Should fail with ZeroDivisionError."""
    rid = glue.start_job_run(JobName=REVENUE_JOB)["JobRunId"]
    print(f"  pipeline 1 (Glue revenue) started: {rid}")
    deadline = time.time() + 600
    while time.time() < deadline:
        run = glue.get_job_run(JobName=REVENUE_JOB, RunId=rid)["JobRun"]
        if run["JobRunState"] in ("SUCCEEDED", "FAILED", "STOPPED", "ERROR", "TIMEOUT"):
            break
        time.sleep(15)
    return {"state": run["JobRunState"], "error": run.get("ErrorMessage", ""), "run_id": rid}


def trigger_pipeline_2():
    """SFN. Should fail with AccessDenied on sns:Publish."""
    exec_arn = sfn.start_execution(stateMachineArn=STATE_MACHINE_ARN, input="{}")["executionArn"]
    print(f"  pipeline 2 (SFN) started")
    deadline = time.time() + 120
    while time.time() < deadline:
        d = sfn.describe_execution(executionArn=exec_arn)
        if d["status"] in ("SUCCEEDED", "FAILED", "TIMED_OUT", "ABORTED"):
            return {"state": d["status"], "error": d.get("cause") or d.get("error") or "", "execution_arn": exec_arn}
        time.sleep(5)
    return {"state": "TIMEOUT", "error": "did not finish in 120s", "execution_arn": exec_arn}


def trigger_pipeline_3():
    """DynamoDB batch write. Should throttle with ProvisionedThroughputExceededException."""
    items = [{"PutRequest": {"Item": {"customer_id": {"S": f"cust-{i}"}, "delta": {"N": str(i * 100)}}}} for i in range(50)]
    unprocessed = 0
    try:
        resp = ddb.batch_write_item(RequestItems={DDB_TABLE: items})
        unprocessed = len(resp.get("UnprocessedItems", {}).get(DDB_TABLE, []))
        return {"state": "PARTIAL" if unprocessed else "OK", "unprocessed": unprocessed}
    except ClientError as exc:
        return {"state": "FAILED", "error": exc.response["Error"]["Code"], "unprocessed": 0}


def trigger_pipeline_4():
    """Lambda consumer. Should be in Failed state or invoke errors."""
    try:
        config = lambda_client.get_function_configuration(FunctionName=CONSUMER_FN)
        # Invoke directly
        invoke = lambda_client.invoke(FunctionName=CONSUMER_FN, Payload=b'{"Records": []}')
        return {"state": "FunctionError" if invoke.get("FunctionError") else "OK",
                "state_reason": config.get("StateReason", ""), "function_state": config.get("State", "")}
    except ClientError as exc:
        return {"state": "FAILED", "error": exc.response["Error"]["Code"]}


# YOUR CODE: Call each trigger and store its result in TRIGGER_RESULTS.
# TRIGGER_RESULTS should be a dict mapping pipeline_id -> result dict.
# For example:
#   TRIGGER_RESULTS = {
#       "pipeline_1": trigger_pipeline_1(),
#       ...
#   }


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: all four pipelines failed in the expected way.

def validator_1(session):
    results = globals().get("TRIGGER_RESULTS")
    if not isinstance(results, dict):
        return CheckpointResult("fail", error_reason="TRIGGER_RESULTS not set. Call each trigger_pipeline_N() function and assign the dict.")
    expected = {
        "pipeline_1": lambda r: r.get("state") in ("FAILED", "ERROR", "TIMEOUT"),
        "pipeline_2": lambda r: r.get("state") == "FAILED",
        "pipeline_3": lambda r: r.get("state") in ("FAILED", "PARTIAL") or r.get("unprocessed", 0) > 0 or r.get("error") == "ProvisionedThroughputExceededException",
        "pipeline_4": lambda r: r.get("state") in ("FunctionError", "FAILED") or r.get("function_state") in ("Failed", "Pending") or r.get("state_reason", ""),
    }
    for pid, predicate in expected.items():
        r = results.get(pid)
        if not r:
            return CheckpointResult("fail", error_reason=f"Missing trigger result for {pid}")
        if not predicate(r):
            return CheckpointResult("fail", error_reason=f"{pid} did not fail as expected: {r}")
    return CheckpointResult("pass")


check(1, validator_1)


<details><summary>Hint 1 (nudge)</summary>

Run each trigger helper and store the result. The validator checks that the failure mode looks right for each pipeline. You are not scored on diagnostic quality here; the gate is "you observed each failure."

</details>

<details><summary>Hint 2 (direction)</summary>

Pipeline 1 should return state in `{FAILED, ERROR}`. Pipeline 2 should return state `FAILED` with an SNS-related cause string. Pipeline 3 should return at least some unprocessed items or a `ProvisionedThroughputExceededException`. Pipeline 4 should return a function in `Failed` or `Pending` state.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
TRIGGER_RESULTS = {
    "pipeline_1": trigger_pipeline_1(),
    "pipeline_2": trigger_pipeline_2(),
    "pipeline_3": trigger_pipeline_3(),
    "pipeline_4": trigger_pipeline_4(),
}
for pid, r in TRIGGER_RESULTS.items():
    print(pid, r)
```

</details>


**Wallet check.** First Glue ETL run at the 10-minute minimum: ~$0.15. SFN execution: free. DynamoDB write throttles: free at this volume. Kinesis 1 shard has been alive ~5 minutes: under a cent. Session total so far: ~17 cents.

## Task 2: Fix Pipeline 1 (revenue aggregation) and verify it now succeeds

The Glue ETL log shows a Spark `ArithmeticException`. The fix is in the script: divide-by-zero on the `self-serve-new` cohort whose customer count is 0. Use `NULLIF` (or `case when customer_count = 0 then null else revenue / customer_count end`) so the per-customer revenue lands NULL for the zero-customer cohort instead of crashing.

You read the current script from S3, fix the divide, upload the fixed script, re-run the job.


In [ ]:
# NBVAL_SKIP
# Task 2: Fix the divide-by-zero, redeploy the script, re-run.

current_script = s3.get_object(Bucket=BUCKET_NAME, Key="scripts/revenue.py")["Body"].read().decode("utf-8")
print("Current script lines around the bug:")
for i, line in enumerate(current_script.splitlines()):
    if "customer_count" in line:
        print(f"  {i+1}: {line}")

# YOUR CODE: rewrite the divide to guard against zero. The simplest fix
# is to replace
#   col("revenue") / col("customer_count")
# with something that produces NULL when customer_count is 0. PySpark
# supports
#   when(col("customer_count") == 0, None).otherwise(col("revenue") / col("customer_count"))
# Use a single-line replacement on the script string and upload it back.

# YOUR CODE: s3.put_object(Bucket=BUCKET_NAME, Key="scripts/revenue.py", Body=fixed.encode("utf-8"))

# YOUR CODE: rerun the job with glue.start_job_run(JobName=REVENUE_JOB)
# and poll until JobRunState in ("SUCCEEDED", "FAILED").

# Once rerun completes, store the final state in REVENUE_RERUN_STATE.


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: revenue job SUCCEEDED and revenue_daily has rows.

def validator_2(session):
    state = globals().get("REVENUE_RERUN_STATE")
    if state != "SUCCEEDED":
        return CheckpointResult("fail", error_reason=f"REVENUE_RERUN_STATE is {state!r}; expected SUCCEEDED. Did you upload the fixed script before starting the job?")
    # Query the Iceberg table.
    WG = f"labrun-{LAB_SLUG}-wg"
    try:
        qid = athena.start_query_execution(
            QueryString=f"SELECT count(*) FROM {DATABASE_NAME}.{REVENUE_TABLE}",
            WorkGroup=WG,
            QueryExecutionContext={"Database": DATABASE_NAME},
        )["QueryExecutionId"]
        deadline = time.time() + 60
        while time.time() < deadline:
            ex = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
            if ex["Status"]["State"] == "SUCCEEDED":
                rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
                count = int(rows[1]["Data"][0]["VarCharValue"])
                if count < 4:
                    return CheckpointResult("fail", error_reason=f"{REVENUE_TABLE} has {count} rows; expected at least 4 (one per cohort).")
                return CheckpointResult("pass")
            if ex["Status"]["State"] in ("FAILED", "CANCELLED"):
                return CheckpointResult("error", error_reason=f"Athena query state {ex['Status']['State']}")
            time.sleep(2)
        return CheckpointResult("error", error_reason="Athena query timeout")
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator query failed: {exc}")


check(2, validator_2)


<details><summary>Hint 1 (nudge)</summary>

CloudWatch Logs Insights against `/aws-glue/jobs/output` shows the Python traceback. Read the line where the ArithmeticException is raised.

</details>

<details><summary>Hint 2 (direction)</summary>

The bug is `col("revenue") / col("customer_count")`. `customer_count` is 0 for the `self-serve-new` cohort. Guard with `when(col("customer_count") == 0, None).otherwise(...)` or `col("revenue") / col("customer_count").nullif(0)` (if your Spark version supports nullif).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
fixed = current_script.replace(
    'df.withColumn("revenue_per_customer", col("revenue") / col("customer_count"))',
    'df.withColumn("revenue_per_customer", when(col("customer_count") == 0, None).otherwise(col("revenue") / col("customer_count")))'
)
# Make sure the import for `when` is there too:
fixed = fixed.replace(
    "from pyspark.sql.functions import col, lit",
    "from pyspark.sql.functions import col, lit, when"
)
s3.put_object(Bucket=BUCKET_NAME, Key="scripts/revenue.py", Body=fixed.encode("utf-8"))

rid = glue.start_job_run(JobName=REVENUE_JOB)["JobRunId"]
deadline = time.time() + 600
while time.time() < deadline:
    state = glue.get_job_run(JobName=REVENUE_JOB, RunId=rid)["JobRun"]["JobRunState"]
    if state in ("SUCCEEDED", "FAILED", "STOPPED", "ERROR", "TIMEOUT"):
        REVENUE_RERUN_STATE = state
        break
    time.sleep(15)
```

</details>


**Wallet check.** Second Glue ETL run at the 10-minute minimum: another ~$0.15. Athena scan for the validator: a fraction of a cent. Session total so far: ~32 cents.

## Task 3: Fix Pipeline 2 (Step Functions IAM regression) and verify the workflow succeeds

The SFN execution history shows the `PublishRevenueAlert` state failed with `States.TaskFailed` and the error message `User: arn:aws:sts::*:assumed-role/labrun-oncall-pipeline-triage-sfn-role/... is not authorized to perform: SNS:Publish`. The role has `logs:*` and nothing else. Fix: add `sns:Publish` on the topic ARN to the role's inline policy. Then restart the execution.


In [ ]:
# NBVAL_SKIP
# Task 3: Add sns:Publish to the SFN role, rerun, expect SUCCEEDED.

# YOUR CODE: Read the current inline policy with iam.get_role_policy and
# add sns:Publish on the SNS_TOPIC_ARN. Then put_role_policy with the
# updated document. IAM updates propagate in 5-10 seconds.

# YOUR CODE: Restart the execution with sfn.start_execution and poll
# until SUCCEEDED or FAILED. Store the final status in SFN_RERUN_STATUS.

# Wait ~10s for IAM propagation before restarting.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: SFN execution SUCCEEDED.

def validator_3(session):
    status = globals().get("SFN_RERUN_STATUS")
    if status != "SUCCEEDED":
        return CheckpointResult("fail", error_reason=f"SFN_RERUN_STATUS is {status!r}; expected SUCCEEDED. Verify the inline policy now grants sns:Publish on the topic ARN.")
    # Confirm an SNS publish actually fired via CloudWatch metric.
    cw = boto3.client("cloudwatch", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
    try:
        from datetime import datetime, timedelta, timezone
        resp = cw.get_metric_statistics(
            Namespace="AWS/SNS", MetricName="NumberOfMessagesPublished",
            Dimensions=[{"Name": "TopicName", "Value": SNS_TOPIC}],
            StartTime=datetime.now(timezone.utc) - timedelta(minutes=10),
            EndTime=datetime.now(timezone.utc), Period=60, Statistics=["Sum"],
        )
        total = sum(p["Sum"] for p in resp.get("Datapoints", []))
        if total < 1:
            return CheckpointResult("fail", error_reason=f"SFN SUCCEEDED but NumberOfMessagesPublished on {SNS_TOPIC} is {total}. Did the Publish step actually run?")
    except ClientError:
        # CloudWatch metric not yet available is acceptable; SFN succeeded is the primary signal.
        pass
    return CheckpointResult("pass")


check(3, validator_3)


<details><summary>Hint 1 (nudge)</summary>

The SFN execution history's failed state carries the AWS error message. Read it before guessing.

</details>

<details><summary>Hint 2 (direction)</summary>

The role's inline policy is missing `sns:Publish`. Add a new statement granting `sns:Publish` on the SNS topic ARN (or, simpler for a lab, `Resource: "*"` since the role is scoped to this lab anyway).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
iam.put_role_policy(
    RoleName=SFN_ROLE,
    PolicyName="labrun-sfn-inline",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["logs:*"], "Resource": "*"},
            {"Effect": "Allow", "Action": ["sns:Publish"], "Resource": SNS_TOPIC_ARN},
        ],
    }),
)
time.sleep(10)
exec_arn = sfn.start_execution(stateMachineArn=STATE_MACHINE_ARN, input="{}")["executionArn"]
deadline = time.time() + 120
while time.time() < deadline:
    d = sfn.describe_execution(executionArn=exec_arn)
    if d["status"] in ("SUCCEEDED", "FAILED", "TIMED_OUT", "ABORTED"):
        SFN_RERUN_STATUS = d["status"]
        break
    time.sleep(5)
```

</details>


**Wallet check.** SFN executions are free at this volume. SNS Publish: free for the first 1M per month. Kinesis ticked another few minutes: under a cent. Session total so far: ~33 cents.

## Task 4: Write the handoff note for the two pipelines you did NOT fix

Pipelines 3 and 4 stay broken. Your handoff note tells the incoming engineer what you saw, what you believe the root cause is, and what to do next. The next engineer is going to read this note while waking up; clarity beats completeness.

Required sections (H2): "Pipeline 3 status", "Pipeline 3 suspected root cause", "Pipeline 3 recommended next step", "Pipeline 4 status", "Pipeline 4 suspected root cause", "Pipeline 4 recommended next step". Each body must be non-empty (at least 50 characters).

Write the markdown to `s3://<bucket>/portfolio/handoff.md`. The portfolio prefix survives cleanup; you keep the artifact even after the lab tears everything else down.


In [ ]:
# NBVAL_SKIP
# Task 4: Write the handoff markdown.

HANDOFF_TEMPLATE = '''# Handoff: pipelines 3 + 4 still broken

## Pipeline 3 status

DynamoDB table `labrun-oncall-pipeline-triage-ddb-table` is throttling writes. The customer-snapshot batch wrote 50 items in a single batch_write_item call against a 1 RCU / 1 WCU table; AWS returned UnprocessedItems for most of them.

## Pipeline 3 suspected root cause

Table provisioned at 1 WCU. The batch writer assumes burst capacity which was exhausted by retries earlier this morning.

## Pipeline 3 recommended next step

Switch the table to on-demand capacity mode (`aws dynamodb update-table --table-name labrun-oncall-pipeline-triage-ddb-table --billing-mode PAY_PER_REQUEST`). On-demand absorbs the burst without provisioning math, and at this volume the cost difference is pennies. Permanent fix; do not raise WCU and call it done.

## Pipeline 4 status

Lambda function `labrun-oncall-pipeline-triage-consumer` is in `Failed` state. Invocations return FunctionError; the configuration shows `StateReasonCode = InvalidConfiguration`.

## Pipeline 4 suspected root cause

Consumer role trust policy trusts `glue.amazonaws.com` instead of `lambda.amazonaws.com`. The Friday commit by Sam (per the incident channel) intended to add the Glue trust for a future cross-service call but accidentally replaced the lambda trust.

## Pipeline 4 recommended next step

Update the trust policy to include both `lambda.amazonaws.com` (primary; required for Lambda to assume the role) and optionally `glue.amazonaws.com` (if Sam's original intent was multi-service). Then re-create the function or wait for the next configuration update to retry.
'''

# YOUR CODE: customize the body, then write to s3://BUCKET/portfolio/handoff.md
# You can use the template verbatim if it matches your observations.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: handoff.md present, all required H2 sections, non-empty bodies.

REQUIRED_SECTIONS_HANDOFF = [
    "Pipeline 3 status",
    "Pipeline 3 suspected root cause",
    "Pipeline 3 recommended next step",
    "Pipeline 4 status",
    "Pipeline 4 suspected root cause",
    "Pipeline 4 recommended next step",
]


def _section_body(markdown, section_title):
    pattern = re.compile(rf"^##\s+{re.escape(section_title)}\s*$", re.MULTILINE)
    match = pattern.search(markdown)
    if not match:
        return None
    start = match.end()
    next_h2 = re.search(r"^##\s+", markdown[start:], re.MULTILINE)
    end = start + next_h2.start() if next_h2 else len(markdown)
    return markdown[start:end].strip()


def validator_4(session):
    try:
        body = s3.get_object(Bucket=BUCKET_NAME, Key="portfolio/handoff.md")["Body"].read().decode("utf-8")
    except ClientError:
        return CheckpointResult("fail", error_reason="portfolio/handoff.md not found in S3. Did you upload it?")
    missing = []
    too_short = []
    for sec in REQUIRED_SECTIONS_HANDOFF:
        text = _section_body(body, sec)
        if text is None:
            missing.append(sec)
        elif len(text) < 50:
            too_short.append(f"{sec} ({len(text)} chars)")
    if missing:
        return CheckpointResult("fail", error_reason=f"Missing H2 section(s): {', '.join(missing)}")
    if too_short:
        return CheckpointResult("fail", error_reason=f"Section(s) under 50 chars: {', '.join(too_short)}")
    return CheckpointResult("pass")


check(4, validator_4)


<details><summary>Hint 1 (nudge)</summary>

The validator parses markdown for the six exact H2 headings. Each section needs a non-trivial body (at least 50 characters). Write what you observed, not what you would have guessed.

</details>

<details><summary>Hint 2 (direction)</summary>

Use the HANDOFF_TEMPLATE as a starting point. Customize the wording so it reflects what your trigger results actually showed. Upload via `s3.put_object`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
s3.put_object(Bucket=BUCKET_NAME, Key="portfolio/handoff.md", Body=HANDOFF_TEMPLATE.encode("utf-8"))
```

</details>


**Wallet check.** Free. S3 put + small markdown file. Session total still: ~33 cents.

## Task 5: Write the post-mortem

The post-mortem is the interview portfolio piece. It is what your manager reads tomorrow and what hiring managers read six months from now. Keep it tight: 150-400 words. Four required H2 sections: Timeline, Root cause, Blast radius, Follow-ups.

Write to `s3://<bucket>/portfolio/postmortem.md`. It survives cleanup.

Voice: factual, no blame, specific times. Reread the incident channel markdown if you need a timeline anchor.


In [ ]:
# NBVAL_SKIP
# Task 5: Write the post-mortem.

POSTMORTEM_TEMPLATE = '''# Post-mortem: four-pipeline incident, 2026-05-15

## Timeline

- 08:47 CFO reports revenue dashboard at zero.
- 08:50 Status page (auto) reports all services operational. Disagreement noted.
- 08:52 finance-billing-alerts reports throughput exceeded on customer-deltas (pipeline 3).
- 08:53 monitoring-bot reports SFN execution failed at PublishRevenueAlert state (pipeline 2).
- 09:01 platform-ops reports Lambda consumer in Failed state since 02:00 (pipeline 4).
- 09:10 Triage begins. Two priority-1 pipelines (revenue ETL, SFN alerting) fixed within 60 minutes. Two pipelines (DynamoDB throttle, Lambda trust policy) handed off with documented next steps.

## Root cause

Two of the four breaks are independent regressions in code committed late Friday. Pipeline 1 (Glue ETL revenue) had an unguarded divide-by-zero in the per-cohort revenue calculation; the self-serve-new cohort had zero customers yesterday. Pipeline 2 (SFN) had an IAM regression: the role's inline policy granted logs but not sns:Publish. Pipelines 3 (DynamoDB throttle) and 4 (Lambda trust policy) are documented in the handoff.

## Blast radius

Revenue dashboard read zero for one day (visible to executive team). Operational alerting was dark for one day (no revenue-drop notifications fired). Customer-delta backlog: ~2500 unwritten rows. Lambda consumer's downstream Kinesis events accumulated on the stream but did not breach the 24h retention.

## Follow-ups

1. Add CI checks for the divide-by-zero pattern (linter rule rejects `a / b` without a guard).
2. Require sns:Publish in the SFN role template; add to the platform IAM module.
3. Move DynamoDB tables off provisioned by default; new tables ship PAY_PER_REQUEST.
4. Add a test that asserts the Lambda trust policy includes lambda.amazonaws.com before deploy.
5. Reconcile the status page with the actual data pipeline health (the disconnect is the trust-erosion problem).
'''

# YOUR CODE: customize or use verbatim, then upload to portfolio/postmortem.md.


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: post-mortem present, required sections, word count 150-400.

REQUIRED_SECTIONS_PM = ["Timeline", "Root cause", "Blast radius", "Follow-ups"]


def validator_5(session):
    try:
        body = s3.get_object(Bucket=BUCKET_NAME, Key="portfolio/postmortem.md")["Body"].read().decode("utf-8")
    except ClientError:
        return CheckpointResult("fail", error_reason="portfolio/postmortem.md not found in S3.")
    for sec in REQUIRED_SECTIONS_PM:
        if _section_body(body, sec) is None:
            return CheckpointResult("fail", error_reason=f"Missing H2 section: {sec}")
        if len(_section_body(body, sec)) < 20:
            return CheckpointResult("fail", error_reason=f"Section {sec!r} body is too short (<20 chars).")
    word_count = len(re.findall(r"\b\w+\b", body))
    if not (150 <= word_count <= 400):
        return CheckpointResult("fail", error_reason=f"Word count is {word_count}; required range 150-400.")
    return CheckpointResult("pass")


check(5, validator_5)


<details><summary>Hint 1 (nudge)</summary>

Required H2 sections: Timeline, Root cause, Blast radius, Follow-ups. Word count 150-400.

</details>

<details><summary>Hint 2 (direction)</summary>

Start with the timeline (you can lift it from the incident channel markdown). The root-cause section names the two pipelines you fixed. Blast radius states user-visible impact. Follow-ups are the four-or-five actions that would prevent recurrence.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
s3.put_object(Bucket=BUCKET_NAME, Key="portfolio/postmortem.md", Body=POSTMORTEM_TEMPLATE.encode("utf-8"))
```

</details>


**Wallet check.** Free. Session total: ~33 cents.

In [ ]:
# NBVAL_SKIP
# Cleanup. Kinesis stream first (critical). Portfolio prefix preserved.

from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter(AwsCleanupAdapter):
    def delete_resource(self, credentials, resource):
        if resource.type == "iceberg_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client("glue", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "s3_bucket":
            # Custom S3 delete that preserves portfolio/ prefix locally
            # (downloads to /tmp first), then empties + deletes the bucket.
            s3c = boto3.client("s3", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            print(f"  preserving portfolio/ before bucket delete")
            try:
                # Stream portfolio objects into print output for the student.
                listing = s3c.list_objects_v2(Bucket=resource.id, Prefix="portfolio/")
                for o in listing.get("Contents", []):
                    body = s3c.get_object(Bucket=resource.id, Key=o["Key"])["Body"].read().decode("utf-8")
                    print(f"  PORTFOLIO ARTIFACT: {o['Key']} ({len(body)} chars). Copy this content if you want it preserved past cleanup.")
                    print(body)
            except ClientError as exc:
                print(f"  portfolio preservation skipped: {exc}")
            # Empty + delete bucket
            try:
                while True:
                    listing = s3c.list_objects_v2(Bucket=resource.id)
                    objs = listing.get("Contents", [])
                    if not objs:
                        break
                    s3c.delete_objects(Bucket=resource.id, Delete={"Objects": [{"Key": o["Key"]} for o in objs]})
                    if not listing.get("IsTruncated"):
                        break
                s3c.delete_bucket(Bucket=resource.id)
            except ClientError:
                pass
            return
        return super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        if resource.type == "iceberg_table":
            return False
        return super().describe_resource(credentials, resource)


# Athena workgroup teardown (it is created by setup; we delete it inline here).
try:
    athena.delete_work_group(WorkGroup=f"labrun-{LAB_SLUG}-wg", RecursiveDeleteOption=True)
    print("  athena workgroup deleted")
except Exception:
    pass

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

print()
print("Cleanup complete.")
critical_count = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_count = len(CLEANUP_MANIFEST) - critical_count
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
failed = len(result.warnings) + len(result.orphans)
print(f"Resources that failed to delete: {failed}")
if failed > 0:
    print("If failed > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
    for w in result.warnings:
        print(f"  WARNING: {w}")
    for o in result.orphans:
        print(f"  ORPHAN: {o}")

cleanup(status=result.status)
if result.status == "dirty":
    sys.exit(1)


**Session total: $1.20 to $2.50.** Glue ETL (two runs) and Kinesis shard-hour are the line items. The chaos lab is moderately priced; the four broken pipelines were small on purpose. Your handoff and post-mortem are preserved in print output from the cleanup cell; download them before closing the notebook.

## These are not graded. They are for you.

**1. Which two pipelines did you fix and why did you pick those two?** Walk through the prioritization. Revenue first or alerting first? What changes if the CFO's deadline is one hour from now versus tomorrow?

**2. Pipeline 3 (DynamoDB throttle) was a documented handoff.** What changes if pipeline 3 were writing customer PII (e.g., GDPR-scoped personal data) rather than internal metrics? Would you have fixed it inside the 75-minute window?

## Exam-Style Practice

**Question 1.** You triage four simultaneous pipeline failures: a $50k/day revenue pipeline, a $5/day internal metrics pipeline, a customer-facing PII export, and a developer-tooling pipeline. Which do you fix first?

A) The revenue pipeline, because revenue is what the business measures.

B) The PII export, because customer-facing data plus regulatory exposure outranks revenue alone.

C) The internal metrics pipeline, because it is the cheapest to fix and clears the backlog quickly.

D) The dev-tooling pipeline, because unblocking engineers compounds across all other fixes.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. Revenue is important, but a PII export that is breaking GDPR or HIPAA is both customer-facing and regulator-facing. The downside is a fine that dwarfs one day of revenue plus reputational damage.
- B) Right. Business impact has two dimensions: dollar value and risk exposure. PII outranks revenue because the regulatory exposure compounds (customer trust + fine + audit). Senior DEs name this trade-off explicitly.
- C) Wrong. "Cheap to fix" is not a triage criterion at 9 AM Monday. Fix order is impact-first, not cost-first.
- D) Wrong. Dev tooling has compounding benefit but is rarely customer-facing. It does not preempt revenue or PII.

</details>

**Question 2.** A pre-broken pipeline raises `States.TaskFailed` on a Step Functions Task that calls SNS Publish. The execution role has `logs:*` attached. What is the single most-likely fix?

A) Restart the execution; SFN sometimes glitches on first invocation.

B) Add `sns:Publish` on the topic ARN to the execution role's inline policy.

C) Switch the SFN integration pattern from sync (`arn:aws:states:::sns:publish`) to async.

D) Raise the SFN execution timeout from 60s to 300s.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. SFN is deterministic; a glitch on first invocation does not produce `States.TaskFailed` with an IAM-style error message. Restarting reproduces the same failure.
- B) Right. `States.TaskFailed` on an SNS Publish step almost always indicates the SFN execution role lacks `sns:Publish` on the topic. The error message in the execution history names the exact AWS error.
- C) Wrong. The sync/async integration pattern affects how SFN waits for the Task to complete. Async would not surface IAM errors any differently; the AccessDenied happens before the integration pattern matters.
- D) Wrong. Timeout matters when the Task is slow. SFN failed before timing out; IAM rejected the call immediately.

</details>
